In [22]:
# imports
from dotenv import load_dotenv
import pandas as pd
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

# load in the .env variables
load_dotenv()

True

In [19]:
claim_data = pd.read_csv("data/CLAIMS.csv")
text_data = pd.read_csv("data/zenodo/publications_tdm.csv")

# reduce text_data to doi which have a match in claim_data
text_data = text_data[text_data["DOI"].isin(claim_data["doi"])]
text_data.head()

,ASK ID,DOI,Title,Abstract,Full-text
215,294803094,10.1002/ece3.5217,Bees and flowers: How to feed an invasive beet...,Invasive species may exploit a wide range of f...,6422 | Ecology and Evolution. 2019;9:6422–...
270,554076691,10.1002/ece3.9493,How does invasion degree shape alpha and beta ...,Freshwater ecosystems appear more vulnerable t...,https://helda.helsinki.fiHow does invasion deg...
312,147103499,10.1002/ecs2.2097,Impact of an invasive alien plant on litter de...,Invasive alien plant effects on ecosystem func...,Impact of an invasive alien plant on litter de...
325,322700873,10.1002/ecs2.2987,Niche change analysis as a tool to inform mana...,Significant progress has been made in providin...,"METHODS, TOOLS, AND TECHNOLOGIESNiche change a..."
1981,533431057,10.1007/s10530-021-02644-y,Parasite DNA detection in water samples enhanc...,Invasive species can facilitate the spread of ...,ORIGINAL PAPERParasite DNA detection in water ...


In [23]:
#Convert DataFrame rows into LangChain Document objects
documents = []
for _, row in text_data.iterrows():
    doc = Document(
        page_content=row["Full-text"],
        metadata={"source": row["DOI"]}
    )
    documents.append(doc)

# Initialize Text Splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len
)

# Chunk the documents (this preserves your metadata for each chunk!)
chunked_docs = text_splitter.split_documents(documents)

# Initialize Embeddings and Store in Vector DB (Chroma)
# Note: Ensure you have your OPENAI_API_KEY environment variable set
embeddings = OpenAIEmbeddings()

vector_store = Chroma.from_documents(
    documents=chunked_docs,
    embedding=embeddings,
    persist_directory="./chroma_db"  # Saves the database locally to disk
)

print(f"Successfully chunked and stored {len(chunked_docs)} chunks.")


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Successfully chunked and stored 3768 chunks.


In [ ]:
# Wie viele der 67 Belegsätze findet die Suche, wenn die Anfrage die Aussage ist?
# Geholt werden die k ähnlichsten Chunks. Ein Treffer heißt: der Belegsatz
# steht als zusammenhängender Text in einem dieser Chunks.
k = 5

# Dieselbe Datenbank wie in Zelle 2, diesmal nur zum Lesen.
# OpenAIEmbeddings bettet gleich die Suchanfrage ein, mit demselben Modell
# wie die Chunks, sonst sind die Vektoren nicht vergleichbar.
vector_store = Chroma(
    persist_directory="./chroma_db",
    embedding_function=OpenAIEmbeddings(),
)

# Alle gespeicherten Chunks holen, gruppiert nach DOI. Das ist nicht die Suche.
# Damit lässt sich hinterher sagen, ob ein verfehlter Belegsatz im Paper
# überhaupt in einem einzelnen Chunk steht oder von einer Grenze zerschnitten wurde.
stored = vector_store.get(include=["documents", "metadatas"])
chunks_by_doi = {}
for text, meta in zip(stored["documents"], stored["metadatas"]):
    chunks_by_doi.setdefault(meta["source"], []).append(text)

hits = 0
in_paper_but_missed = []
split_across_chunks = []

for _, row in claim_data.iterrows():
    # Suchanfrage ist row["statement"], der englische Satz aus CLAIMS.csv.
    # Nicht der Belegsatz und nicht der ganze Claim als Zeile.
    # similarity_search bettet diesen Satz ein und vergleicht den Vektor mit den
    # Chunk-Vektoren. Zurück kommen die k Chunks mit dem kleinsten Abstand.
    #
    # filter={"source": row["doi"]} lässt vor diesem Vergleich alle Chunks weg,
    # deren Metadaten-DOI nicht die DOI dieser Zeile ist. Geprüft werden also
    # nur die Chunks des einen Papers, nicht die anderen 47.
    docs = vector_store.similarity_search(
        row["statement"],
        k=k,
        filter={"source": row["doi"]},
    )
    evidence = row["evidence"]
    # Wortgenauer Vergleich, keine Ähnlichkeit. Der Belegsatz muss in einem
    # der fünf Chunk-Texte als Teilstring stehen.
    if any(evidence in doc.page_content for doc in docs):
        hits += 1
        continue
    # Nicht in den Top k. Liegt der Satz wenigstens in irgendeinem Chunk dieses Papers?
    paper_chunks = chunks_by_doi[row["doi"]]
    if any(evidence in chunk for chunk in paper_chunks):
        in_paper_but_missed.append(row["claim_id"])
    else:
        split_across_chunks.append(row["claim_id"])

n = len(claim_data)
print(f"{hits} / {n} evidence spans are in the top {k} chunks of the same paper ({hits / n:.0%}).")
print(f"{len(in_paper_but_missed)} are in that paper, but not in the top {k}: {', '.join(in_paper_but_missed)}")
print(f"{len(split_across_chunks)} are cut by a chunk boundary, so no single chunk contains them: {', '.join(split_across_chunks)}")

In [ ]:
import json
import time
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI

# Alle 67 Aussagen. Das Modell sieht nur die Aussage und die fünf Chunks.
# Gold-Label und Belegsatz bleiben außerhalb des Prompts und dienen nur dem Vergleich.
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
system = SystemMessage(content="""You check one claim against five passages from the same paper.
Reply with JSON only, no markdown:
{"label": "supported" | "not supported" | "unclear", "rationale": "<one or two sentences>"}

supported: a passage reports the paper's own result and that result matches the claim.
not supported: the paper tested the claim and a passage reports the opposite or no effect.
unclear: the passages bear on the claim but do not settle it.
Use only these passages.""")

rows = []
for _, row in claim_data.iterrows():
    docs = vector_store.similarity_search(
        row["statement"],
        k=k,
        filter={"source": row["doi"]},
    )
    evidence_in_topk = any(row["evidence"] in doc.page_content for doc in docs)
    passages = "\n\n".join(
        f"Passage {i}:\n{doc.page_content}" for i, doc in enumerate(docs, start=1)
    )
    reply = None
    for attempt in range(3):
        try:
            reply = llm.invoke([
                system,
                HumanMessage(content=f"Claim:\n{row['statement']}\n\n{passages}"),
            ])
            break
        except Exception:
            if attempt == 2:
                raise
            time.sleep(2)
    text = reply.content.strip()
    if text.startswith("```"):
        text = text.split("\n", 1)[-1].rsplit("```", 1)[0].strip()
    parsed = json.loads(text)
    rows.append({
        "claim_id": row["claim_id"],
        "doi": row["doi"],
        "gold": row["label"],
        "predicted": parsed["label"],
        "evidence_in_topk": evidence_in_topk,
        "match": parsed["label"] == row["label"],
        "rationale": parsed["rationale"],
    })
    print(f"{row['claim_id']} {row['label']} -> {parsed['label']}", flush=True)

label_check = pd.DataFrame(rows)
label_check.to_csv("results/predictions.csv", index=False)
correct = int(label_check["match"].sum())
print(f"{correct} / {len(label_check)} labels match the gold label. Saved results/predictions.csv.")
print(pd.crosstab(label_check["gold"], label_check["predicted"], margins=True))
label_check

In [ ]:
import json
import time
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI

# Dieselben Vorhersagen wie in predictions.csv. Entfernt wird der Belegsatz nur dort,
# wo er in den fünf Chunks stand. Verglichen wird mit predicted, nicht mit dem Gold-Label.
baseline = pd.read_csv("results/predictions.csv")
baseline = baseline[baseline["evidence_in_topk"] == True].copy()
claims = claim_data.set_index("claim_id")

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
system = SystemMessage(content="""You check one claim against five passages from the same paper.
Reply with JSON only, no markdown:
{"label": "supported" | "not supported" | "unclear", "rationale": "<one or two sentences>"}

supported: a passage reports the paper's own result and that result matches the claim.
not supported: the paper tested the claim and a passage reports the opposite or no effect.
unclear: the passages bear on the claim but do not settle it.
Use only these passages.""")

rows = []
for _, row in baseline.iterrows():
    statement = claims.loc[row["claim_id"], "statement"]
    evidence = claims.loc[row["claim_id"], "evidence"]
    docs = vector_store.similarity_search(
        statement,
        k=k,
        filter={"source": row["doi"]},
    )
    # In jedem der fünf Chunks den wortgenauen Belegsatz streichen. Durch die
    # Überlappung kann derselbe Satz in zwei Chunks stehen.
    passages = []
    removed = 0
    for i, doc in enumerate(docs, start=1):
        text = doc.page_content
        if evidence in text:
            text = text.replace(evidence, " ")
            removed += 1
        passages.append(f"Passage {i}:\n{text}")
    if removed == 0:
        raise RuntimeError(f"{row['claim_id']}: evidence span was not in the retrieved chunks")
    reply = None
    for attempt in range(3):
        try:
            reply = llm.invoke([
                system,
                HumanMessage(content=f"Claim:\n{statement}\n\n" + "\n\n".join(passages)),
            ])
            break
        except Exception:
            if attempt == 2:
                raise
            time.sleep(2)
    text = reply.content.strip()
    if text.startswith("```"):
        text = text.split("\n", 1)[-1].rsplit("```", 1)[0].strip()
    parsed = json.loads(text)
    flipped = parsed["label"] != row["predicted"]
    rows.append({
        "claim_id": row["claim_id"],
        "doi": row["doi"],
        "gold": row["gold"],
        "predicted": row["predicted"],
        "predicted_without_evidence": parsed["label"],
        "flipped": flipped,
        "rationale_without_evidence": parsed["rationale"],
    })
    print(f"{row['claim_id']} {row['predicted']} -> {parsed['label']}{'  flipped' if flipped else ''}", flush=True)

intervention = pd.DataFrame(rows)
intervention.to_csv("results/intervention.csv", index=False)
flips = int(intervention["flipped"].sum())
print(f"{flips} / {len(intervention)} labels changed after the evidence span was removed.")
print(pd.crosstab(intervention["gold"], intervention["flipped"], margins=True))
intervention

In [ ]:
# Satz-SHAP für die Claims, deren Label nach Intervention 1 supported geblieben ist.
# Spieler sind die Sätze der fünf Chunks ohne Belegsatz. Der Wert ist log P(supported).
# KernelSHAP, Hintergrund ist der leere Text. Pro Claim die Samples max(2 * Sätze, 40).
from scripts.sentence_shap import run_sentence_shap

shap_sentences = run_sentence_shap()
print(f"{shap_sentences['claim_id'].nunique()} claims, {len(shap_sentences)} sentences. Saved shap_sentences.csv.")
shap_sentences.groupby("claim_id")["shap_value"].max().sort_values(ascending=False)

In [ ]:
# Intervention 2: die bis zu drei Sätze mit dem höchsten positiven SHAP-Wert entfernen.
# Verglichen wird mit dem Label nach Intervention 1.
from scripts.sentence_shap import run_intervention_2

intervention_2 = run_intervention_2()
flips = int(intervention_2["flipped"].sum())
print(f"{flips} / {len(intervention_2)} labels changed. Saved intervention_2.csv.")
print(pd.crosstab(intervention_2["gold"], intervention_2["flipped"], margins=True))
intervention_2

In [ ]:
# Dieselben 32 Claims, SHAP-Werte aus shap_sentences.csv.
# Neu gelabelt wird nur nach Entfernen der 1, 5, 7 und 10 wichtigsten Sätze.
# Die Bedingung mit 3 Sätzen kommt aus intervention_2.csv.
from scripts.sentence_shap import results_table, run_shap_removals

removals = run_shap_removals((1, 3, 5, 7, 10))
results = results_table(removals)
results

In [ ]:
# Ablation: niedrigste SHAP-Sätze und zufällige Sätze, dieselben Anzahlen.
# Die höchsten SHAP-Sätze bleiben die Zeilen aus shap_removal.csv.
from scripts.sentence_shap import results_table, run_ablations

ablation = run_ablations((1, 3, 5, 7, 10))
results = results_table(ablation)
results

In [ ]:
# Fünf Stichproben pro Bedingung, Temperatur 0.7, top-p 0.9.
# Agreement: Anteil der Stichproben mit dem Mehrheitslabel.
# Confidence: mittlere Wahrscheinlichkeit des Label-Tokens auf den Stichproben dieser Mehrheit.
from scripts.sample_labels import run_sampling

sampled_results = run_sampling()
sampled_results

In [ ]:
# Ähnlichkeit des Belegsatzes zu den entfernten Sätzen, dieselben Mengen wie in der Ablation.
# Embeddings sind dieselben wie im Retriever. Cosinus innerhalb jeder Aussage.
from scripts.shap_similarity import run_similarity

similarity_results = run_similarity()
similarity_results

In [ ]:
# Dieselben 67 Aussagen und dieselben fünf Stücke, Temperatur 0.
# Im JSON steht die Begründung vor dem Label. Verglichen wird mit predictions.csv.
from scripts.order_intervention import run_order

order_results = run_order()
order_results